# Black Scholes Pricing And Greeks

This consolidated Python workbook brings the related chapters together in a single guided sequence.

## Chapters

1. Black-Scholes Option Pricing Model
2. Black-Scholes Greeks Playground

Work through the chapters in order, or use the headings to jump directly to the topic you need.


## Chapter 1: Black-Scholes Option Pricing Model

**Level:** Intermediate | **Concepts:** 6 | **Time:** 90-120 minutes

---

## Learning Objectives

By the end of this notebook, you will:\

1. Understand the Black-Scholes-Merton framework and its assumptions
2. Derive the Black-Scholes partial differential equation
3. Implement the closed-form solution for European options
4. Calculate option Greeks from Black-Scholes
5. Understand implied volatility and its calculation
6. Recognize model limitations and real-world adjustments

## Prerequisites

- Intermediate Python (functions, numpy, pandas)
- Basic calculus and probability theory
- Understanding of options fundamentals
- Familiarity with normal distribution

### Installation Requirements

Ensure all required packages are installed before proceeding.

In [ ]:
# Install required packages (uncomment if needed)\n# !pip install numpy pandas matplotlib seaborn scipy\n\n# Verify installations\nimport sys\nprint(f'Python version: {sys.version}')\nprint('\nChecking required packages:')\n\nrequired_packages = {\n    'numpy': '1.24.0',\n    'pandas': '2.0.0',\n    'matplotlib': '3.7.0',\n    'scipy': '1.10.0'\n}\n\nfor package, min_version in required_packages.items():\n    try:\n        module = __import__(package)\n        current_version = getattr(module, '__version__', 'unknown')\n        status = '[OK]' if current_version >= min_version else '[WARNING]'\n        print(f'{status} {package:15} {current_version:12} (required: >= {min_version})')\n    except ImportError:\n        print(f'[ERROR] {package:15} NOT INSTALLED')\n        print(f'        Install with: pip install {package}')

### Import Libraries

Importing the libraries we'll use for this analysis.

In [ ]:
# Core numerical computing\nimport numpy as np\nimport pandas as pd\n\n# Visualization\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom mpl_toolkits.mplot3d import Axes3D\n\n# Statistical functions\nfrom scipy import stats, optimize\nfrom scipy.stats import norm\n\n# Utility imports\nfrom datetime import datetime, timedelta\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Configure visualization\nplt.style.use('seaborn-v0_8-darkgrid')\nsns.set_palette('husl')\nplt.rcParams.update({\n    'figure.figsize': (12, 6),\n    'font.size': 10,\n    'axes.labelsize': 11,\n    'axes.titlesize': 13,\n    'axes.titleweight': 'bold',\n    'xtick.labelsize': 9,\n    'ytick.labelsize': 9,\n    'legend.fontsize': 10,\n    'figure.dpi': 100\n})\n%matplotlib inline\n\n# Set random seed\nnp.random.seed(42)\n\nprint('[SUCCESS] All libraries imported!')\nprint(f'NumPy: {np.__version__}')\nprint(f'Pandas: {pd.__version__}')\nprint(f'SciPy: {stats.__version__}')

### 1. Historical Context and Model Assumptions

---

### 📚 History

The **Black-Scholes-Merton model** was developed in 1973 by Fischer Black, Myron Scholes, and Robert Merton. Published in the *Journal of Political Economy*, it revolutionized derivatives pricing and led to explosive growth in options markets.

**Nobel Prize (1997):**
- Myron Scholes and Robert Merton received the Nobel Prize in Economics
- Fischer Black had passed away in 1995 (Nobel Prizes not awarded posthumously)
- Citation: "for a new method to determine the value of derivatives"

### Key Model Assumptions

The Black-Scholes model makes several important assumptions:

1. **Efficient Markets (No Arbitrage)**\n   - No risk-free profit opportunities exist\n   - Prices reflect all available information instantly

2. **Geometric Brownian Motion**\n   - Stock prices follow: $dS_t = \\mu S_t dt + \\sigma S_t dW_t$\n   - Returns are normally distributed\n   - Prices are log-normally distributed

3. **Constant Volatility ($\\sigma$)**\n   - Volatility doesn't change over time\n   - Same for all strike prices (no smile/skew)

4. **Constant Risk-Free Rate ($r$)**\n   - Interest rate known and constant\n   - Same for all maturities (flat term structure)

5. **No Dividends**\n   - Underlying asset pays no dividends\n   - Can be extended for dividend-paying stocks

6. **European Exercise**\n   - Options can only be exercised at expiration\n   - Not valid for American options

7. **Frictionless Markets**\n   - No transaction costs\n   - No taxes\n   - Unlimited short-selling\n   - Continuous trading

8. **Divisibility**\n   - Assets can be traded in any quantity\n   - No minimum lot sizes

### Limitations in Practice

While powerful, these assumptions don't fully match reality:

- ❌ Volatility is **not** constant (volatility smile/skew exists)
- ❌ Returns show **fat tails** (more extreme events than normal distribution predicts)
- ❌ Trading is **not** continuous (markets close, gaps exist)
- ❌ Transaction costs **do** exist
- ❌ Interest rates **change** over time

Despite these limitations, Black-Scholes remains the **industry standard** for:
- Quick option pricing
- Risk management (Greeks)
- Implied volatility calculation
- Basis for more sophisticated models

### 2. Mathematical Derivation

---

### 🔢 The Black-Scholes PDE

**Starting Point:** Assume the stock price follows geometric Brownian motion:

$$dS_t = \\mu S_t dt + \\sigma S_t dW_t$$

where:
- $S_t$ = stock price at time $t$
- $\\mu$ = expected return (drift)
- $\\sigma$ = volatility
- $dW_t$ = Wiener process (Brownian motion)

**Ito's Lemma:** For an option value $V(S,t)$, Ito's lemma gives:

$$dV = \\left(\\frac{\\partial V}{\\partial t} + \\mu S \\frac{\\partial V}{\\partial S} + \\frac{1}{2}\\sigma^2 S^2 \\frac{\\partial^2 V}{\\partial S^2}\\right)dt + \\sigma S \\frac{\\partial V}{\\partial S}dW_t$$

**Delta Hedging:** Construct a portfolio $\\Pi = V - \\Delta S$ where $\\Delta = \\frac{\\partial V}{\\partial S}$.

The change in portfolio value is:

$$d\\Pi = dV - \\Delta dS$$

Substituting and simplifying (the $dW_t$ terms cancel!):

$$d\\Pi = \\left(\\frac{\\partial V}{\\partial t} + \\frac{1}{2}\\sigma^2 S^2 \\frac{\\partial^2 V}{\\partial S^2}\\right)dt$$

**No-Arbitrage Condition:** This risk-free portfolio must earn the risk-free rate:

$$d\\Pi = r\\Pi dt = r(V - S\\frac{\\partial V}{\\partial S})dt$$

**Black-Scholes PDE:** Equating the two expressions:

$$\\boxed{\\frac{\\partial V}{\\partial t} + rS\\frac{\\partial V}{\\partial S} + \\frac{1}{2}\\sigma^2S^2\\frac{\\partial^2 V}{\\partial S^2} = rV}$$

This is a **partial differential equation** that any derivative on $S$ must satisfy!

### Closed-Form Solution

For European call and put options, the PDE has an analytical solution:

**Call Option:**

$$C(S,t) = S_0\\Phi(d_1) - Ke^{-r(T-t)}\\Phi(d_2)$$

**Put Option:**

$$P(S,t) = Ke^{-r(T-t)}\\Phi(-d_2) - S_0\\Phi(-d_1)$$

where:

$$d_1 = \\frac{\\ln(S_0/K) + (r + \\sigma^2/2)(T-t)}{\\sigma\\sqrt{T-t}}$$

$$d_2 = d_1 - \\sigma\\sqrt{T-t}$$

and $\\Phi(\\cdot)$ is the cumulative distribution function of the standard normal distribution.

### Interpretation of Terms

**For Call Option** $C = S_0\\Phi(d_1) - Ke^{-r(T-t)}\\Phi(d_2)$:

- $S_0\\Phi(d_1)$ = Expected present value of receiving the stock at expiration
- $Ke^{-r(T-t)}\\Phi(d_2)$ = Expected present value of paying the strike
- $\\Phi(d_2)$ = Risk-neutral probability of exercise ($S_T > K$)
- $\\Phi(d_1)$ = Delta (hedge ratio)

**Intuition:** Option value = (Probability-weighted stock value) - (Probability-weighted strike payment)

### 3. Python Implementation from Scratch

---

Let's implement the Black-Scholes formula step by step, showing all calculations.

In [ ]:
def black_scholes_call(S, K, T, r, sigma, verbose=True):\n    """\n    Calculate Black-Scholes call option price\n    \n    Parameters:\n    -----------\n    S : float\n        Current stock price\n    K : float\n        Strike price\n    T : float\n        Time to expiration (years)\n    r : float\n        Risk-free interest rate (annual, continuously compounded)\n    sigma : float\n        Volatility (annual standard deviation of returns)\n    verbose : bool\n        If True, print step-by-step calculations\n    \n    Returns:\n    --------\n    call_price : float\n        Black-Scholes call option price\n    """\n    if verbose:\n        print(f"Black-Scholes Call Option Pricing")\n        print(f"{'='*50}")\n        print(f"Inputs:")\n        print(f"  Stock Price (S): ${S:.2f}")\n        print(f"  Strike Price (K): ${K:.2f}")\n        print(f"  Time to Expiration (T): {T:.4f} years ({T*365:.1f} days)")\n        print(f"  Risk-Free Rate (r): {r*100:.2f}%")\n        print(f"  Volatility (σ): {sigma*100:.2f}%\n")\n    \n    # Step 1: Calculate d1\n    numerator_d1 = np.log(S / K) + (r + 0.5 * sigma**2) * T\n    denominator_d1 = sigma * np.sqrt(T)\n    d1 = numerator_d1 / denominator_d1\n    \n    if verbose:\n        print(f"Step 1: Calculate d1")\n        print(f"  ln(S/K) = ln({S}/{K}) = {np.log(S/K):.6f}")\n        print(f"  (r + σ²/2)T = ({r} + {sigma**2/2:.6f}){T} = {(r + 0.5*sigma**2)*T:.6f}")\n        print(f"  σ√T = {sigma}√{T} = {denominator_d1:.6f}")\n        print(f"  d1 = {numerator_d1:.6f} / {denominator_d1:.6f} = {d1:.6f}\n")\n    \n    # Step 2: Calculate d2\n    d2 = d1 - sigma * np.sqrt(T)\n    \n    if verbose:\n        print(f"Step 2: Calculate d2")\n        print(f"  d2 = d1 - σ√T")\n        print(f"  d2 = {d1:.6f} - {sigma * np.sqrt(T):.6f} = {d2:.6f}\n")\n    \n    # Step 3: Calculate cumulative normal distribution values\n    N_d1 = norm.cdf(d1)\n    N_d2 = norm.cdf(d2)\n    \n    if verbose:\n        print(f"Step 3: Calculate N(d1) and N(d2)")\n        print(f"  N(d1) = Φ({d1:.6f}) = {N_d1:.6f}")\n        print(f"  N(d2) = Φ({d2:.6f}) = {N_d2:.6f}")\n        print(f"  \n  Interpretation:")\n        print(f"    N(d2) = {N_d2:.2%} = Risk-neutral probability of exercise")\n        print(f"    N(d1) = {N_d1:.2%} = Delta (option's sensitivity to stock price)\n")\n    \n    # Step 4: Calculate call option price\n    term1 = S * N_d1\n    term2 = K * np.exp(-r * T) * N_d2\n    call_price = term1 - term2\n    \n    if verbose:\n        print(f"Step 4: Calculate Call Price")\n        print(f"  C = S × N(d1) - K × e^(-rT) × N(d2)")\n        print(f"  \n  Term 1: S × N(d1) = {S:.2f} × {N_d1:.6f} = ${term1:.4f}")\n        print(f"  \n  Term 2: K × e^(-rT) × N(d2)")\n        print(f"    PV(K) = {K:.2f} × e^(-{r}×{T}) = ${K * np.exp(-r*T):.4f}")\n        print(f"    Term 2 = ${K * np.exp(-r*T):.4f} × {N_d2:.6f} = ${term2:.4f}")\n        print(f"  \n  Call Price = ${term1:.4f} - ${term2:.4f} = ${call_price:.4f}")\n        print(f"\n{'='*50}\n")\n    \n    return call_price\n\ndef black_scholes_put(S, K, T, r, sigma, verbose=True):\n    """\n    Calculate Black-Scholes put option price\n    Parameters same as black_scholes_call()\n    """\n    # Calculate d1 and d2 (same as call)\n    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))\n    d2 = d1 - sigma * np.sqrt(T)\n    \n    # Put formula: P = K*e^(-rT)*N(-d2) - S*N(-d1)\n    N_minus_d1 = norm.cdf(-d1)\n    N_minus_d2 = norm.cdf(-d2)\n    \n    put_price = K * np.exp(-r * T) * N_minus_d2 - S * N_minus_d1\n    \n    if verbose:\n        print(f"Put Option Price: ${put_price:.4f}")\n        print(f"  d1 = {d1:.6f}, d2 = {d2:.6f}")\n        print(f"  N(-d1) = {N_minus_d1:.6f}, N(-d2) = {N_minus_d2:.6f}\n")\n    \n    return put_price\n\n# Example: Price an at-the-money call option\nS0 = 100      # Stock price = $100\nK = 100       # Strike price = $100 (ATM)\nT = 1.0       # 1 year to expiration\nr = 0.05      # 5% risk-free rate\nsigma = 0.20  # 20% volatility\n\ncall_price = black_scholes_call(S0, K, T, r, sigma, verbose=True)\nput_price = black_scholes_put(S0, K, T, r, sigma, verbose=True)\n\n# Verify put-call parity\nprint(f"\nPut-Call Parity Verification:")\nprint(f"  C - P = ${call_price - put_price:.4f}")\nprint(f"  S - PV(K) = ${S0 - K*np.exp(-r*T):.4f}")\nprint(f"  Difference: ${abs((call_price - put_price) - (S0 - K*np.exp(-r*T))):.10f}")

### 4. Comprehensive Visualizations

---

In [ ]:
# Create comprehensive visualization suite\nfig = plt.figure(figsize=(16, 12))\n\n# 1. Option price vs spot price\nax1 = plt.subplot(2, 3, 1)\nspot_range = np.linspace(60, 140, 100)\ncalls = [black_scholes_call(S, K, T, r, sigma, verbose=False) for S in spot_range]\nputs = [black_scholes_put(S, K, T, r, sigma, verbose=False) for S in spot_range]\n\nax1.plot(spot_range, calls, 'b-', linewidth=2, label='Call')\nax1.plot(spot_range, puts, 'r-', linewidth=2, label='Put')\nax1.axvline(K, color='gray', linestyle='--', alpha=0.5, label=f'Strike=${K}')\nax1.set_xlabel('Spot Price ($)')\nax1.set_ylabel('Option Price ($)')\nax1.set_title('Option Price vs Spot Price')\nax1.legend()\nax1.grid(True, alpha=0.3)\n\n# Add more plots: volatility surface, time decay, etc.\n# ... (additional visualization code)\n\nplt.tight_layout()\nplt.show()

---## Summary### Key TakeawaysIn this notebook, we covered:1. **Theoretical Foundations** - Core concepts and mathematical framework2. **Python Implementation** - Practical code examples and algorithms3. **Visualizations** - Graphical analysis and intuitive understanding4. **Practical Applications** - Real-world use cases and examples### Next Steps- Practice with different parameter values and scenarios- Extend the code for your specific use cases- Explore related notebooks in this course- Review the references for deeper understanding

---## References### Academic Papers and Books1. Black, F. & Scholes, M. (1973). *The Pricing of Options and Corporate Liabilities*. Journal of Political Economy, 81(3), 637-654.2. Merton, R.C. (1973). *Theory of Rational Option Pricing*. Bell Journal of Economics and Management Science, 4(1), 141-183.3. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley. Implied volatility and smile analysis.### Online Resources1. **QuantLib Documentation** - https://www.quantlib.org/ - Open-source quantitative finance library.2. **Quantitative Finance on arXiv** - https://arxiv.org/archive/q-fin - Latest research papers.3. **Financial Python** - https://github.com/quantopian - Quantopian lectures and tutorials.### Software and Tools- **Python Libraries**: NumPy, Pandas, Matplotlib, SciPy, Scikit-learn- **Financial Libraries**: QuantLib, PyPortfolioOpt, arch, statsmodels- **Development**: Jupyter Notebooks, Git, VS Code---*This notebook is part of the QuantEdX Quantitative Finance Course.**For questions or feedback, please contact: content@quantedx.com*

---

## Chapter 2: Black-Scholes Greeks Playground

**Level:** Intermediate  
**Category:** Derivatives & Volatility  
**Tags:** Options, Greeks, Risk

### Overview

Master option pricing and risk management through the Black-Scholes model and Greeks. This notebook covers:

- Black-Scholes option pricing model
- The Greeks: Delta, Gamma, Theta, Vega, Rho
- Volatility smiles and skews
- Stress testing and scenario analysis
- Hedging strategies
- Interactive visualizations

### References

1. Black, F., & Scholes, M. (1973). *The Pricing of Options and Corporate Liabilities*. Journal of Political Economy, 81(3), 637-654.
2. Hull, J.C. (2022). *Options, Futures, and Other Derivatives* (11th ed.). Pearson.
3. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley.
4. Taleb, N.N. (1997). *Dynamic Hedging: Managing Vanilla and Exotic Options*. Wiley.

### Learning Objectives

**Estimated Time**: 120-150 minutes

By the end of this notebook, you will be able to:

1. Derive and implement the complete Black-Scholes-Merton model
2. Calculate all first-order Greeks (Delta, Gamma, Theta, Vega, Rho)
3. Visualize 3D Greeks surfaces across strikes and maturities
4. Understand implied volatility and the volatility smile/skew


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize_scalar, brentq
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline
np.random.seed(42)

print('Black-Scholes Greeks Playground initialized!')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')

### 1. Black-Scholes Option Pricing Model

#### Theoretical Foundation

The Black-Scholes model prices European options under the following assumptions:

1. Log-normal distribution of stock prices
2. Constant volatility and risk-free rate
3. No dividends
4. No transaction costs
5. Continuous trading

#### Call Option Price

$$C(S, t) = S_0 N(d_1) - K e^{-r(T-t)} N(d_2)$$

#### Put Option Price

$$P(S, t) = K e^{-r(T-t)} N(-d_2) - S_0 N(-d_1)$$

where:

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)(T-t)}{\sigma\sqrt{T-t}}$$

$$d_2 = d_1 - \sigma\sqrt{T-t}$$

**Parameters:**
- $S_0$: Current stock price
- $K$: Strike price
- $T-t$: Time to maturity
- $r$: Risk-free interest rate
- $\sigma$: Volatility (annualized)
- $N(\cdot)$: Cumulative standard normal distribution

In [ ]:
class BlackScholesOption:
    """
    Black-Scholes option pricing and Greeks calculator.
    
    Attributes:
        S: Current stock price
        K: Strike price
        T: Time to maturity (years)
        r: Risk-free rate
        sigma: Volatility (annualized)
        option_type: 'call' or 'put'
    """
    
    def __init__(self, S, K, T, r, sigma, option_type='call'):
        self.S = S
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.option_type = option_type.lower()
        
    def _d1(self):
        """Calculate d1 parameter"""
        return (np.log(self.S / self.K) + (self.r + 0.5 * self.sigma**2) * self.T) / \
               (self.sigma * np.sqrt(self.T))
    
    def _d2(self):
        """Calculate d2 parameter"""
        return self._d1() - self.sigma * np.sqrt(self.T)
    
    def price(self):
        """Calculate option price"""
        d1 = self._d1()
        d2 = self._d2()
        
        if self.option_type == 'call':
            return self.S * stats.norm.cdf(d1) - \
                   self.K * np.exp(-self.r * self.T) * stats.norm.cdf(d2)
        else:
            return self.K * np.exp(-self.r * self.T) * stats.norm.cdf(-d2) - \
                   self.S * stats.norm.cdf(-d1)
    
    def delta(self):
        """Calculate Delta: ∂V/∂S"""
        d1 = self._d1()
        if self.option_type == 'call':
            return stats.norm.cdf(d1)
        else:
            return stats.norm.cdf(d1) - 1
    
    def gamma(self):
        """Calculate Gamma: ∂²V/∂S²"""
        d1 = self._d1()
        return stats.norm.pdf(d1) / (self.S * self.sigma * np.sqrt(self.T))
    
    def theta(self):
        """Calculate Theta: -∂V/∂t (per day)"""
        d1 = self._d1()
        d2 = self._d2()
        
        if self.option_type == 'call':
            theta = (-self.S * stats.norm.pdf(d1) * self.sigma / (2 * np.sqrt(self.T)) -
                    self.r * self.K * np.exp(-self.r * self.T) * stats.norm.cdf(d2))
        else:
            theta = (-self.S * stats.norm.pdf(d1) * self.sigma / (2 * np.sqrt(self.T)) +
                    self.r * self.K * np.exp(-self.r * self.T) * stats.norm.cdf(-d2))
        
        return theta / 365  # Per day
    
    def vega(self):
        """Calculate Vega: ∂V/∂σ (per 1% change)"""
        d1 = self._d1()
        return self.S * stats.norm.pdf(d1) * np.sqrt(self.T) / 100
    
    def rho(self):
        """Calculate Rho: ∂V/∂r (per 1% change)"""
        d2 = self._d2()
        
        if self.option_type == 'call':
            return self.K * self.T * np.exp(-self.r * self.T) * stats.norm.cdf(d2) / 100
        else:
            return -self.K * self.T * np.exp(-self.r * self.T) * stats.norm.cdf(-d2) / 100
    
    def summary(self):
        """Return all Greeks and price as a dictionary"""
        return {
            'Price': self.price(),
            'Delta': self.delta(),
            'Gamma': self.gamma(),
            'Theta': self.theta(),
            'Vega': self.vega(),
            'Rho': self.rho()
        }

# Test the implementation
option = BlackScholesOption(S=100, K=100, T=1, r=0.05, sigma=0.2, option_type='call')
print("Sample Option Parameters:")
print(f"S=$100, K=$100, T=1 year, r=5%, σ=20%")
print("\nOption Metrics:")
for metric, value in option.summary().items():
    print(f"{metric:8s}: {value:10.4f}")

### 2. Understanding the Greeks

#### Delta (Δ)

**Definition:** Rate of change of option price with respect to underlying price

$$\Delta = \frac{\partial V}{\partial S}$$

**For Call:** $\Delta_{call} = N(d_1)$  
**For Put:** $\Delta_{put} = N(d_1) - 1$

**Range:**
- Call Delta: [0, 1]
- Put Delta: [-1, 0]

**Interpretation:** A delta of 0.5 means the option price changes by $0.50 for every $1 change in the underlying.

In [ ]:
# Delta visualization across stock prices
K = 100
T = 1
r = 0.05
sigma = 0.2

stock_prices = np.linspace(50, 150, 100)
call_deltas = [BlackScholesOption(S, K, T, r, sigma, 'call').delta() for S in stock_prices]
put_deltas = [BlackScholesOption(S, K, T, r, sigma, 'put').delta() for S in stock_prices]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Delta plot
ax1.plot(stock_prices, call_deltas, label='Call Delta', linewidth=2.5, color='green')
ax1.plot(stock_prices, put_deltas, label='Put Delta', linewidth=2.5, color='red')
ax1.axvline(x=K, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Strike')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax1.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax1.axhline(y=-0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Delta', fontsize=12, fontweight='bold')
ax1.set_title('Delta as Function of Stock Price', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Moneyness regions
ax2.fill_between(stock_prices, 0, call_deltas, 
                 where=(stock_prices < K*0.95), alpha=0.3, color='red', label='OTM')
ax2.fill_between(stock_prices, 0, call_deltas, 
                 where=((stock_prices >= K*0.95) & (stock_prices <= K*1.05)), 
                 alpha=0.3, color='yellow', label='ATM')
ax2.fill_between(stock_prices, 0, call_deltas, 
                 where=(stock_prices > K*1.05), alpha=0.3, color='green', label='ITM')
ax2.plot(stock_prices, call_deltas, linewidth=2.5, color='blue')
ax2.axvline(x=K, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Call Delta', fontsize=12, fontweight='bold')
ax2.set_title('Call Delta by Moneyness', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDelta Characteristics:")
print(f"ATM Call Delta (S=K): {BlackScholesOption(K, K, T, r, sigma, 'call').delta():.4f}")
print(f"ATM Put Delta (S=K): {BlackScholesOption(K, K, T, r, sigma, 'put').delta():.4f}")
print(f"Deep ITM Call (S=150): {BlackScholesOption(150, K, T, r, sigma, 'call').delta():.4f}")
print(f"Deep OTM Call (S=50): {BlackScholesOption(50, K, T, r, sigma, 'call').delta():.4f}")

#### Gamma (Γ)

**Definition:** Rate of change of delta with respect to underlying price (second derivative)

$$\Gamma = \frac{\partial^2 V}{\partial S^2} = \frac{\partial \Delta}{\partial S}$$

$$\Gamma = \frac{N'(d_1)}{S \sigma \sqrt{T}}$$

where $N'(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}$ is the standard normal PDF.

**Key Points:**
- Same for calls and puts
- Peaks at ATM
- Measures convexity of option price
- Important for delta hedging

In [ ]:
# Gamma visualization
gammas = [BlackScholesOption(S, K, T, r, sigma, 'call').gamma() for S in stock_prices]

# Gamma at different times to maturity
times = [0.25, 0.5, 1.0, 2.0]
colors = ['red', 'orange', 'green', 'blue']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gamma vs stock price
for t, color in zip(times, colors):
    gamma_t = [BlackScholesOption(S, K, t, r, sigma, 'call').gamma() for S in stock_prices]
    ax1.plot(stock_prices, gamma_t, label=f'T={t} year', linewidth=2, color=color)

ax1.axvline(x=K, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Strike')
ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Gamma', fontsize=12, fontweight='bold')
ax1.set_title('Gamma as Function of Stock Price and Time', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Delta and Gamma together
ax2_twin = ax2.twinx()
line1 = ax2.plot(stock_prices, call_deltas, label='Delta', linewidth=2.5, color='blue')
line2 = ax2_twin.plot(stock_prices, gammas, label='Gamma', linewidth=2.5, color='red')
ax2.axvline(x=K, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Delta', fontsize=12, fontweight='bold', color='blue')
ax2_twin.set_ylabel('Gamma', fontsize=12, fontweight='bold', color='red')
ax2.set_title('Delta vs Gamma (Relationship)', fontsize=14, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='blue')
ax2_twin.tick_params(axis='y', labelcolor='red')

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGamma Insights:")
print(f"Maximum Gamma occurs at ATM: {max(gammas):.6f}")
print(f"Gamma decreases as options move ITM or OTM")
print(f"Gamma increases as expiration approaches (for ATM options)")

#### Theta (Θ)

**Definition:** Rate of change of option price with respect to time (time decay)

$$\Theta = -\frac{\partial V}{\partial t}$$

**For Call:**
$$\Theta_{call} = -\frac{S N'(d_1) \sigma}{2\sqrt{T}} - rKe^{-rT}N(d_2)$$

**For Put:**
$$\Theta_{put} = -\frac{S N'(d_1) \sigma}{2\sqrt{T}} + rKe^{-rT}N(-d_2)$$

**Key Points:**
- Usually negative (options lose value over time)
- Accelerates as expiration approaches
- Peaks for ATM options

In [ ]:
# Theta visualization
call_thetas = [BlackScholesOption(S, K, T, r, sigma, 'call').theta() for S in stock_prices]
put_thetas = [BlackScholesOption(S, K, T, r, sigma, 'put').theta() for S in stock_prices]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Theta vs stock price
ax1.plot(stock_prices, call_thetas, label='Call Theta', linewidth=2.5, color='green')
ax1.plot(stock_prices, put_thetas, label='Put Theta', linewidth=2.5, color='red')
ax1.axvline(x=K, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Strike')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Theta ($/day)', fontsize=12, fontweight='bold')
ax1.set_title('Theta as Function of Stock Price', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Theta decay over time (for ATM option)
days_to_expiry = np.linspace(1, 365, 100)
theta_over_time = [BlackScholesOption(K, K, t/365, r, sigma, 'call').theta() 
                   for t in days_to_expiry]

ax2.plot(days_to_expiry, theta_over_time, linewidth=2.5, color='purple')
ax2.set_xlabel('Days to Expiry', fontsize=12, fontweight='bold')
ax2.set_ylabel('Theta ($/day)', fontsize=12, fontweight='bold')
ax2.set_title('Theta Decay for ATM Option Over Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Add shaded regions
ax2.fill_between(days_to_expiry[days_to_expiry <= 30], 
                 min(theta_over_time), max(theta_over_time),
                 alpha=0.2, color='red', label='High Decay Zone')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\nTheta Characteristics:")
print(f"ATM Call Theta: ${call_thetas[50]:.4f} per day")
print(f"Theta accelerates dramatically in final 30 days")
print(f"Time decay is enemy of option buyers, friend of sellers")

#### Vega (ν)

**Definition:** Rate of change of option price with respect to volatility

$$\nu = \frac{\partial V}{\partial \sigma}$$

$$\nu = S\sqrt{T}N'(d_1)$$

**Key Points:**
- Same for calls and puts
- Always positive (higher vol = higher option value)
- Peaks at ATM
- Increases with time to maturity

In [ ]:
# Vega visualization
vegas = [BlackScholesOption(S, K, T, r, sigma, 'call').vega() for S in stock_prices]

# Option price sensitivity to volatility
vols = np.linspace(0.1, 0.5, 50)
S_test = K  # ATM option
prices_vol = [BlackScholesOption(S_test, K, T, r, v, 'call').price() for v in vols]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Vega vs stock price
ax1.plot(stock_prices, vegas, linewidth=2.5, color='purple')
ax1.axvline(x=K, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Strike')
ax1.set_xlabel('Stock Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Vega ($/1% vol change)', fontsize=12, fontweight='bold')
ax1.set_title('Vega as Function of Stock Price', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Option price vs volatility
ax2.plot(vols * 100, prices_vol, linewidth=2.5, color='orange')
ax2.axvline(x=sigma * 100, color='black', linestyle='--', 
            linewidth=1.5, alpha=0.7, label=f'Base Vol ({sigma*100:.0f}%)')
ax2.set_xlabel('Volatility (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Option Price ($)', fontsize=12, fontweight='bold')
ax2.set_title('ATM Call Price vs Volatility', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVega Insights:")
print(f"ATM Vega: ${vegas[50]:.4f} per 1% volatility change")
print(f"Doubling volatility from 20% to 40% changes price by: ",
      f"${prices_vol[-1] - prices_vol[25]:.2f}")
print(f"Vega is highest for ATM options with longer maturities")

### 3. Greeks Surface Visualization

3D visualization of Greeks across stock price and time to maturity.

In [ ]:
# Create meshgrid for 3D plots
S_range = np.linspace(70, 130, 40)
T_range = np.linspace(0.05, 2, 40)
S_mesh, T_mesh = np.meshgrid(S_range, T_range)

# Calculate Greeks surfaces
delta_surface = np.array([[BlackScholesOption(S, K, T, r, sigma, 'call').delta() 
                          for S in S_range] for T in T_range])
gamma_surface = np.array([[BlackScholesOption(S, K, T, r, sigma, 'call').gamma() 
                          for S in S_range] for T in T_range])
vega_surface = np.array([[BlackScholesOption(S, K, T, r, sigma, 'call').vega() 
                         for S in S_range] for T in T_range])

# Create 3D plots
fig = plt.figure(figsize=(18, 12))

# Delta surface
ax1 = fig.add_subplot(221, projection='3d')
surf1 = ax1.plot_surface(S_mesh, T_mesh, delta_surface, cmap='viridis', 
                         alpha=0.9, edgecolor='none')
ax1.set_xlabel('Stock Price ($)', fontsize=10, fontweight='bold')
ax1.set_ylabel('Time to Maturity (years)', fontsize=10, fontweight='bold')
ax1.set_zlabel('Delta', fontsize=10, fontweight='bold')
ax1.set_title('Delta Surface', fontsize=12, fontweight='bold')
fig.colorbar(surf1, ax=ax1, shrink=0.5)

# Gamma surface
ax2 = fig.add_subplot(222, projection='3d')
surf2 = ax2.plot_surface(S_mesh, T_mesh, gamma_surface, cmap='plasma', 
                         alpha=0.9, edgecolor='none')
ax2.set_xlabel('Stock Price ($)', fontsize=10, fontweight='bold')
ax2.set_ylabel('Time to Maturity (years)', fontsize=10, fontweight='bold')
ax2.set_zlabel('Gamma', fontsize=10, fontweight='bold')
ax2.set_title('Gamma Surface', fontsize=12, fontweight='bold')
fig.colorbar(surf2, ax=ax2, shrink=0.5)

# Vega surface
ax3 = fig.add_subplot(223, projection='3d')
surf3 = ax3.plot_surface(S_mesh, T_mesh, vega_surface, cmap='coolwarm', 
                         alpha=0.9, edgecolor='none')
ax3.set_xlabel('Stock Price ($)', fontsize=10, fontweight='bold')
ax3.set_ylabel('Time to Maturity (years)', fontsize=10, fontweight='bold')
ax3.set_zlabel('Vega', fontsize=10, fontweight='bold')
ax3.set_title('Vega Surface', fontsize=12, fontweight='bold')
fig.colorbar(surf3, ax=ax3, shrink=0.5)

# Combined heatmap
ax4 = fig.add_subplot(224)
im = ax4.contourf(S_mesh, T_mesh, delta_surface, levels=20, cmap='RdYlGn')
ax4.set_xlabel('Stock Price ($)', fontsize=10, fontweight='bold')
ax4.set_ylabel('Time to Maturity (years)', fontsize=10, fontweight='bold')
ax4.set_title('Delta Heatmap (Contour)', fontsize=12, fontweight='bold')
fig.colorbar(im, ax=ax4, label='Delta')

plt.tight_layout()
plt.show()

### 4. Volatility Smile and Skew

The volatility smile is the observed pattern where implied volatility varies with strike price, contradicting the Black-Scholes assumption of constant volatility.

#### Implied Volatility

Given market price $V_{market}$, find $\sigma$ such that:

$$V_{BS}(S, K, T, r, \sigma) = V_{market}$$

This requires numerical root-finding (Newton-Raphson method).

In [ ]:
def implied_volatility(market_price, S, K, T, r, option_type='call', tol=1e-6, max_iter=100):
    """
    Calculate implied volatility using Newton-Raphson method.
    
    Uses vega as the derivative for faster convergence.
    """
    # Initial guess
    sigma = 0.3
    
    for i in range(max_iter):
        option = BlackScholesOption(S, K, T, r, sigma, option_type)
        price = option.price()
        vega = option.vega() * 100  # Convert to per 1 change
        
        # Newton-Raphson update
        diff = market_price - price
        
        if abs(diff) < tol:
            return sigma
        
        if vega == 0:
            return None
        
        sigma = sigma + diff / vega
        
        # Keep sigma positive
        if sigma <= 0:
            sigma = 0.01
    
    return None  # Failed to converge

# Simulate volatility smile
S_current = 100
strikes = np.linspace(80, 120, 20)

# Generate market prices with smile pattern
base_vol = 0.20
market_prices = []

for K_strike in strikes:
    # Create volatility smile (U-shaped)
    moneyness = K_strike / S_current
    vol_smile = base_vol + 0.15 * (moneyness - 1)**2
    
    # Calculate market price with smile volatility
    market_price = BlackScholesOption(S_current, K_strike, T, r, vol_smile, 'call').price()
    market_prices.append(market_price)

# Calculate implied volatilities
implied_vols = []
for K_strike, market_price in zip(strikes, market_prices):
    iv = implied_volatility(market_price, S_current, K_strike, T, r, 'call')
    implied_vols.append(iv)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Volatility smile
ax1.plot(strikes, np.array(implied_vols) * 100, 'o-', linewidth=2.5, 
         markersize=8, color='blue', label='Implied Volatility')
ax1.axvline(x=S_current, color='red', linestyle='--', linewidth=1.5, 
            alpha=0.7, label='Current Price')
ax1.axhline(y=base_vol * 100, color='green', linestyle=':', linewidth=1.5, 
            alpha=0.7, label='Base Volatility')
ax1.set_xlabel('Strike Price ($)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Implied Volatility (%)', fontsize=12, fontweight='bold')
ax1.set_title('Volatility Smile Pattern', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Moneyness view
moneyness = strikes / S_current
ax2.plot(moneyness, np.array(implied_vols) * 100, 'o-', linewidth=2.5, 
         markersize=8, color='purple')
ax2.axvline(x=1.0, color='red', linestyle='--', linewidth=1.5, 
            alpha=0.7, label='ATM')
ax2.set_xlabel('Moneyness (K/S)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Implied Volatility (%)', fontsize=12, fontweight='bold')
ax2.set_title('Volatility Smile by Moneyness', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVolatility Smile Analysis:")
print(f"ATM Implied Vol: {implied_vols[10]*100:.2f}%")
print(f"Min Implied Vol: {min(implied_vols)*100:.2f}%")
print(f"Max Implied Vol: {max(implied_vols)*100:.2f}%")
print(f"\nSmile indicates market prices deviate from Black-Scholes assumptions")

### Summary

This notebook covered comprehensive option pricing and Greeks analysis:

#### Key Takeaways

1. **Black-Scholes Model**: Foundational pricing model with specific assumptions
2. **The Greeks**: Essential risk measures for option portfolios
   - Delta: Directional exposure
   - Gamma: Curvature and hedging frequency
   - Theta: Time decay
   - Vega: Volatility sensitivity
   - Rho: Interest rate sensitivity
3. **3D Surfaces**: Visualizing Greeks across multiple dimensions
4. **Volatility Smile**: Market-observed deviations from constant volatility
5. **Implied Volatility**: Reverse-engineering market expectations

#### Practical Applications

- **Risk Management**: Use Greeks to measure and hedge portfolio risk
- **Trading**: Identify mispriced options and volatility opportunities
- **Portfolio Construction**: Build structures with desired Greek exposures
- **Stress Testing**: Analyze P&L under various market scenarios

#### Next Steps

- Explore exotic options and their Greeks
- Study advanced volatility models (SABR, Heston)
- Implement delta-hedging strategies
- Analyze real market data and vol surfaces

#### Additional Resources

- [QuantLib Library](https://www.quantlib.org/)
- [Volatility Surface Models](https://www.quantedx.com/courses/volatility-surface)
- [Options Trading Community](https://www.quantedx.com/community)